# GnitzDB — SQL quickstart: efficient incremental materialized views

**GnitzDB** is a streaming SQL database whose headline feature is **efficient
incremental materialized views**. Every `CREATE VIEW` compiles to a **DBSP**
circuit; when base data changes, gnitz updates the view from just the *delta*
instead of recomputing it. Internally every relation is a **Z-set** — each row
carries an integer *weight* — the motto being *"ZSets all the way down."*

The pattern this notebook follows is the one that matters in practice:

> **Define your views first, then stream data in — every insert is maintained
> incrementally.**

Using plain SQL through the Python bindings, we will:

1. Connect to a running `gnitz-server` over its Unix domain socket.
2. Create two tables (`customers`, `orders`).
3. Define a **JOIN** view, then a **GROUP BY** view layered on top of it.
4. Stream data in and read the continuously-maintained results.
5. Query a table with `SELECT ... WHERE ...`.
6. Insert more data and watch the views update **incrementally**.

> Every statement goes through `conn.execute_sql(...)`. gnitz also has a
> low-level Python *circuit-builder* API — we deliberately do **not** use it
> here; this is a pure-SQL tour.

## 0. Start a server

gnitz is a client/server engine, and the client talks to a `gnitz-server`
process over a **Unix domain socket** (there is no TCP host/port). From the
repository root:

```bash
# Build the server binary and the Python extension (the `gnitz` package).
make server
make pyext

# Launch a server:  gnitz-server <data_dir> <socket_path> [--workers=N]
./gnitz-server /tmp/gnitz-demo-data /tmp/gnitz-demo.sock --workers=4
```

Leave that running in a terminal. The socket path you give the server is the
same path this notebook connects to (override it with the `GNITZ_SOCK`
environment variable).

**Kernel:** make sure this notebook's kernel can `import gnitz`. The simplest
way is to start Jupyter from the extension's environment, e.g.

```bash
cd crates/gnitz-py && uv run jupyter lab
```

In [ ]:
import os
import gnitz

# The server's socket path — must match the path you launched gnitz-server with.
SOCK_PATH = os.environ.get("GNITZ_SOCK", "/tmp/gnitz-demo.sock")

# Tables and views live in a schema. A `public` schema exists out of the box,
# so there is no need to CREATE SCHEMA first.
SCHEMA = "public"

print("will connect to", SOCK_PATH)

## 1. Connect

`gnitz.connect(socket_path)` returns a `Connection`. It is also a context
manager (`with gnitz.connect(path) as conn: ...`); here we keep a long-lived
handle so each cell can be run on its own, and close it at the very end.

In [ ]:
conn = gnitz.connect(SOCK_PATH)
print("connected:", conn)

### (optional) Clean slate

So the notebook can be re-run against the same server, drop anything left over
from a previous run. Views must be dropped before the tables they read, and the
`GROUP BY` view before the join view it sits on. Dropping something that does
not exist simply raises — we ignore that on a fresh server.

In [ ]:
for stmt in (
    "DROP VIEW revenue_by_city",
    "DROP VIEW big_orders",
    "DROP VIEW order_details",
    "DROP TABLE orders",
    "DROP TABLE customers",
):
    try:
        conn.execute_sql(stmt, schema_name=SCHEMA)
    except Exception:
        pass  # didn't exist yet — fine on a fresh server

print("clean slate ready")

## 2. Create two tables

Column types used here are `BIGINT` (64-bit integer) and `TEXT` (string;
`VARCHAR(n)` is accepted too — both map to the same string type). A primary key
must be an integer column. `execute_sql` returns a list with one result dict per
statement; a `CREATE TABLE` yields `{'type': 'TableCreated', 'table_id': ...}`.

In [ ]:
res = conn.execute_sql(
    "CREATE TABLE customers ("
    "  id   BIGINT NOT NULL PRIMARY KEY,"
    "  name TEXT   NOT NULL,"
    "  city TEXT   NOT NULL"
    ")",
    schema_name=SCHEMA,
)
print(res)

res = conn.execute_sql(
    "CREATE TABLE orders ("
    "  id          BIGINT NOT NULL PRIMARY KEY,"
    "  customer_id BIGINT NOT NULL,"
    "  amount      BIGINT NOT NULL"
    ")",
    schema_name=SCHEMA,
)
print(res)

## 3. A view with a JOIN

We define the views **before** inserting data — that is the whole point of
gnitz: the data we stream in next flows through these views incrementally.

Each gnitz view is a single relational *shape* — one of: a filter/projection, a
single JOIN, or a GROUP BY. So we start with the join: pair every order with its
customer. (A join view cannot carry its own `WHERE`/`GROUP BY`; those go in a
view layered on top — that is the next step.)

A `CREATE VIEW` returns `{'type': 'ViewCreated', 'view_id': ...}`; we keep the
id to scan the view later.

In [ ]:
res = conn.execute_sql(
    "CREATE VIEW order_details AS "
    "SELECT customers.name AS name, "
    "       customers.city AS city, "
    "       orders.amount  AS amount "
    "FROM orders "
    "JOIN customers ON orders.customer_id = customers.id",
    schema_name=SCHEMA,
)
order_details_id = res[0]["view_id"]
print(res, "-> view id", order_details_id)

## 4. A view with GROUP BY (layered on the join)

Because JOIN and GROUP BY are different shapes, *join **and** group* is expressed
as a `GROUP BY` view that reads **from the join view**. This one rolls the joined
rows up to revenue per city. `SUM` / `COUNT` / `MIN` / `MAX` are all available;
a `GROUP BY` view's SELECT list may contain only the group column(s) and
aggregates.

In [ ]:
res = conn.execute_sql(
    "CREATE VIEW revenue_by_city AS "
    "SELECT city, "
    "       SUM(amount) AS total, "
    "       COUNT(*)    AS order_count "
    "FROM order_details "
    "GROUP BY city",
    schema_name=SCHEMA,
)
revenue_by_city_id = res[0]["view_id"]
print(res, "-> view id", revenue_by_city_id)

## 5. Stream data in

The views already exist, so every `INSERT` below is processed **incrementally**:
gnitz pushes each delta through the join and the group-by and updates the
materialized views — it never rescans the base tables. `execute_sql` runs DML
too, and an `INSERT` returns `{'type': 'RowsAffected', 'count': N}`.

In [ ]:
res = conn.execute_sql(
    "INSERT INTO customers VALUES "
    "(1, 'Alice', 'Berlin'), "
    "(2, 'Bob',   'Hamburg'), "
    "(3, 'Carol', 'Berlin'), "
    "(4, 'Dave',  'Hamburg'), "
    "(5, 'Erin',  'Munich')",
    schema_name=SCHEMA,
)
print(res)

res = conn.execute_sql(
    "INSERT INTO orders VALUES "
    "(101, 1, 250), "
    "(102, 1, 125), "
    "(103, 2, 400), "
    "(104, 3,  90), "
    "(105, 4, 510), "
    "(106, 5, 300), "
    "(107, 2, 150)",
    schema_name=SCHEMA,
)
print(res)

## 6. Read a view

A view is a maintained **Z-set**, so we read it with `conn.scan(view_id)`. Each
row carries a `.weight`; we keep only rows with **positive net weight** (a
steady-state view row has weight `1`; retractions / ghosts are `<= 0`). A `Row`
supports `row["col"]`, `row.col`, `row[i]`, and `row._asdict()`.

With the data above, `revenue_by_city` should read **Berlin** 465 / 3 orders,
**Hamburg** 1060 / 3 orders, **Munich** 300 / 1 order.

In [ ]:
def view_rows(view_id, cols, sort_key=None):
    # positive-weight rows of a view, projected to `cols` (accessed by name)
    rows = [{c: r[c] for c in cols}
            for r in conn.scan(view_id) if r.weight > 0]
    return sorted(rows, key=sort_key) if sort_key else rows

def show(rows):
    rows = list(rows)
    if not rows:
        print("(no rows)")
        return
    try:  # a DataFrame renders nicely if pandas is installed...
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(rows))
    except Exception:  # ...otherwise fall back to plain prints (no dependency)
        for r in rows:
            print(r)

print("order_details  (the JOIN view):")
show(view_rows(order_details_id, ["name", "city", "amount"],
               sort_key=lambda d: d["name"]))

print("\nrevenue_by_city  (the GROUP BY view):")
show(view_rows(revenue_by_city_id, ["city", "total", "order_count"],
               sort_key=lambda d: d["city"]))

## 7. `SELECT ... WHERE ...`

A direct `SELECT` is a **primary-key point lookup** — `WHERE <pk> = <value>` (or
no `WHERE` at all); the result is a `Rows` dict whose `"rows"` is a scannable
result. To filter on a **non-key** column, express the predicate as a **view**
and scan it — that is gnitz's continuous-query model.

(The `big_orders` view below is created *after* its data already exists: gnitz
backfills a new view from the current rows, then maintains it incrementally. So
view-first is not required for *correctness* — it is simply how you get
incremental maintenance from the very first row.)

In [ ]:
# (a) Primary-key point lookup — supported directly.
res = conn.execute_sql("SELECT * FROM customers WHERE id = 1", schema_name=SCHEMA)
assert res[0]["type"] == "Rows"
print("customer 1:", [r._asdict() for r in res[0]["rows"]])

# (b) Filter on a non-key column -> define a view, then scan it.
res = conn.execute_sql(
    "CREATE VIEW big_orders AS SELECT * FROM orders WHERE amount >= 300",
    schema_name=SCHEMA,
)
big_orders_id = res[0]["view_id"]

print("\norders with amount >= 300:")
show(view_rows(big_orders_id, ["id", "customer_id", "amount"],
               sort_key=lambda d: d["id"]))

## 8. Incremental maintenance — the payoff

This is the whole reason gnitz exists. Insert one more order and **re-scan the
same view**: gnitz updates `revenue_by_city` from the single-row delta, with no
recompute. Munich goes from 300 / 1 order to **1000 / 2 orders**.

In [ ]:
conn.execute_sql("INSERT INTO orders VALUES (108, 5, 700)", schema_name=SCHEMA)

print("revenue_by_city after the new Munich order:")
show(view_rows(revenue_by_city_id, ["city", "total", "order_count"],
               sort_key=lambda d: d["city"]))

## 9. Close

`Connection` is also a context manager, so `with gnitz.connect(...) as conn:`
would close it automatically. Closing is idempotent.

In [ ]:
conn.close()
print("closed")